In [1]:
import numpy as np
import pandas as pd


In [3]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [7]:
movies = movies.merge(credits,on='title')
#Merged movies and credits

In [10]:
'''Useful columns :
genres
id
keywords
title
overview
cast
crew
'''
movies = movies[['movie_id','title','genres','keywords','overview','cast','crew']]

In [ ]:
#Now we need to find all the tags associated with a particular movie
#We do this by merging overview , keywords , cast , crew , genres into one singular tags column
#We shall only consider top 3 cast members and directors of each movie and merge everything else into one big string


In [11]:
movies.isnull().sum()

,0
movie_id,0
title,0
genres,0
keywords,0
overview,3
cast,0
crew,0


In [12]:
movies.dropna(inplace=True)

In [14]:
movies.duplicated().sum()

np.int64(0)

In [20]:
import ast
def convert(obj):
  L = []
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L



In [25]:
movies['genres'] = movies['genres'].apply(convert)

In [26]:
movies['keywords'] = movies['keywords'].apply(convert)

In [28]:
def convert3(obj):
  L = []
  counter = 0
  for i in ast.literal_eval(obj):
    if counter != 3:
      L.append(i['name'])
      counter+=1
    else:
      break
  return L

In [29]:
movies['cast'] = movies['cast'].apply(convert3)

In [34]:
def convert_crew(obj):
  L = []
  for i in ast.literal_eval(obj):
    if i['job'] == 'Director':
      L.append(i['name'])
      break
  return L

In [36]:
movies['crew'] = movies['crew'].apply(convert_crew)

In [38]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [40]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [45]:
movies['tags'] = movies['overview'] + movies['keywords'] + movies['cast'] + movies['crew']

In [46]:
new_df = movies[['movie_id','title','tags']]

In [49]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

/tmp/ipython-input-140/3089450492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [51]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


/tmp/ipython-input-140/4103658309.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just wants to play his guitar and ...
4805,72766,Newlyweds,a newlywed couple's honeymoon is upended by th...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduces a dedic..."
4807,126186,Shanghai Calling,when ambitious new york attorney sam is sent t...


In [65]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')


In [66]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [59]:
import nltk

In [62]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()


In [61]:
def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [64]:
new_df['tags'] = new_df['tags'].apply(stem)

/tmp/ipython-input-140/3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [72]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [91]:
def recommend(movie):
  movie_index = new_df[new_df['title'] == movie].index[0]
  distances = similarity[movie_index]
  movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

  for i in movies_list:
      print(new_df.iloc[i[0]].title)



